day 02:


Setup: run it first in every session

In [1]:
from pathlib import Path
import os

# Find data/lite in the current folder or one of its parent folders.
start = Path.cwd().resolve()
kit_root = next(
    (folder for folder in (start, *start.parents)
     if (folder / "data" / "lite").is_dir()),
    None,
)

if kit_root is None:
    raise FileNotFoundError(
        f"Could not find data/lite from {start}. "
        "Open the notebook from within the project, or set the project path explicitly."
    )

os.chdir(kit_root)
DATA_DIR = kit_root / "data" / "lite"

print("Working in:", kit_root)
print("Dataset folder:", DATA_DIR)
print("Contents:", [path.name for path in sorted(DATA_DIR.iterdir())[:20]])

Working in: /home/aang/aang-brain/career/acud/oasis_datakit
Dataset folder: /home/aang/aang-brain/career/acud/oasis_datakit/data/lite
Contents: ['air_quality.csv', 'air_quality.csv:Zone.Identifier', 'bin_fill.csv', 'bin_fill.csv:Zone.Identifier', 'bms_readings.csv', 'bms_readings.csv:Zone.Identifier', 'camera_status.csv', 'camera_status.csv:Zone.Identifier', 'collection_trips.csv', 'collection_trips.csv:Zone.Identifier', 'electricity_meters.csv', 'electricity_meters.csv:Zone.Identifier', 'gov_services.csv', 'gov_services.csv:Zone.Identifier', 'incidents.csv', 'incidents.csv:Zone.Identifier', 'network_pressure.csv', 'network_pressure.csv:Zone.Identifier', 'noise.csv', 'noise.csv:Zone.Identifier']


Which files does Sunday's report need?

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

tr_raw = pd.read_csv("data/lite/traffic_counts.csv",
                     parse_dates=["timestamp"])
el_raw = pd.read_csv("data/lite/electricity_meters.csv",
                     parse_dates=["timestamp"])
sensors = pd.read_csv("reference/sensors.csv")
cal = pd.read_csv("reference/calendar.csv", parse_dates=["date"])

report = []        # one row per issue
print(f"{len(tr_raw):,} traffic rows")
print(f"{len(el_raw):,} electricity rows")


481,560 traffic rows
234,160 electricity rows


Can we reuse the Day 1 model on any version of the data?


In [3]:
def usual_alarms(t):
    """Day 1 model: alarm when an hour is below half of usual"""
    t = t.assign(date=t["timestamp"].dt.normalize(),
                 hour=t["timestamp"].dt.hour)
    t = t.merge(cal, on="date")
    t = t[(t["is_weekend"] == 0) & (t["is_holiday"] == 0)]
    g = ["sensor_id", "date", "hour", "is_ramadan"]
    h = t.groupby(g, as_index=False)["vehicle_count"].sum()
    h["usual"] = (h.groupby(["sensor_id", "hour", "is_ramadan"])
                   ["vehicle_count"].transform("median"))
    h["alarm"] = h["vehicle_count"] < 0.5 * h["usual"]
    return h


How many alarms does the model raise on the raw traffic data?

In [4]:
raw_h = usual_alarms(tr_raw)
print(len(raw_h), "hourly readings ·",
      raw_h["alarm"].sum(), "alarms")


75579 hourly readings · 21 alarms


What does the same model say when it learns from clean data?


In [5]:
key = ["sensor_id", "timestamp"]
c = tr_raw.drop_duplicates(subset=key)
c = c[c["sensor_id"].isin(sensors["sensor_id"])]   # registered
real = c["timestamp"].dt.tz_localize("Africa/Cairo",
            nonexistent="NaT", ambiguous="NaT").notna()
c = c[real].dropna(subset=["vehicle_count"])

hour = c["timestamp"].dt.floor("h")
n = (c.groupby(["sensor_id", hour])["vehicle_count"]
      .transform("size"))
clean_h = usual_alarms(c[n == 4])      # complete hours only
print(len(clean_h), "hourly readings ·",
      clean_h["alarm"].sum(), "alarms")


71584 hourly readings · 10 alarms


D3 · Garbage in, garbage out

Of the alarms each version raises, how many point at the real problem: TRF-023 stuck on 10 June?

In [6]:
def score(h):
    a = h[h["alarm"]]
    real = ((a["sensor_id"] == "TRF-023")
            & (a["date"] == "2025-06-10")).sum()
    return {"alarms": len(a), "real": real,
            "false": len(a) - real,
            "precision_pct": round(100 * real / len(a))}

display(pd.DataFrame({"raw data": score(raw_h),
                      "clean data": score(clean_h)}).T)


,alarms,real,false,precision_pct
raw data,21,10,11,48
clean data,10,10,0,100


D4a · Count the readings

How many readings sit behind each false alarm?

In [7]:
false = raw_h[raw_h["alarm"]
              & (raw_h["sensor_id"] != "TRF-023")]
ok = tr_raw.dropna(subset=["vehicle_count"])
per_hour = ok.groupby(["sensor_id",
                       ok["timestamp"].dt.floor("h")]).size()
false = false.assign(readings=[
    per_hour.get((s, d + pd.Timedelta(hours=h)), 0)
    for s, d, h in zip(false["sensor_id"], false["date"],
                       false["hour"])])
print(len(false), "false alarms")


11 false alarms


D4b · What they have in common


What do the false alarms on the raw data have in common?


In [8]:
display(false[["sensor_id", "date", "hour", "vehicle_count",
               "usual", "readings"]].head(6))
print("false alarms with fewer than 4 readings:",
      (false["readings"] < 4).sum(), "of", len(false))


,sensor_id,date,hour,vehicle_count,usual,readings
10350,TRF-006,2025-05-14,14,739.0,1510.0,2
20718,TRF-012,2025-03-26,14,316.0,636.5,2
22384,TRF-013,2025-03-18,4,74.0,150.5,2
27497,TRF-015,2025-06-19,5,116.0,332.0,2
28196,TRF-016,2025-04-07,8,902.0,1810.0,2
39200,TRF-022,2025-03-27,20,726.0,1457.0,2


false alarms with fewer than 4 readings: 11 of 11


D5a · Apply the four rules

What does each suggested rule do to the raw file?


In [9]:
s = tr_raw.sort_values(["sensor_id", "timestamp"])
z = s.groupby("sensor_id")["vehicle_count"].transform(
    lambda x: (x - x.mean()) / x.std())
rules = {   # four rules a chatbot typically suggests
    "fill missing counts with 0": s.fillna({"vehicle_count": 0}),
    "remove outliers (|z| > 3)": s[~(z.abs() > 3)],
    "drop exact duplicate rows": s.drop_duplicates(),
    "fill missing with the previous reading": s.assign(
        vehicle_count=s.groupby("sensor_id")["vehicle_count"]
                       .ffill()),
}
for name, data in rules.items():
    print(f"{name}: {len(data):,} rows")


fill missing counts with 0: 481,560 rows
remove outliers (|z| > 3): 481,369 rows
drop exact duplicate rows: 479,164 rows
fill missing with the previous reading: 481,560 rows


D5b · Score each rule

Which of the four rules does the data support?


In [10]:
tests = {"raw data": score(raw_h)}
for name, data in rules.items():
    tests[name] = score(usual_alarms(data))
tests["clean data (D2)"] = score(clean_h)
display(pd.DataFrame(tests).T)


,alarms,real,false,precision_pct
raw data,21,10,11,48
fill missing counts with 0,21,10,11,48
remove outliers (|z| > 3),22,10,12,45
drop exact duplicate rows,19,10,9,53
fill missing with the previous reading,10,10,0,100
clean data (D2),10,10,0,100


Inspect 1 · Size and gaps

How big is electricity_meters, and what is missing?

In [11]:
import pandas as pd
from IPython.display import display

el = pd.read_csv("data/lite/electricity_meters.csv",
                 parse_dates=["timestamp"])
print(el.shape)
print(el.isna().sum())

(234160, 3)
building_id       0
timestamp         0
kwh            3340
dtype: int64


Inspect 2 · Impossible values

Which kWh values cannot be true for one building in one hour?



In [12]:
print(el["kwh"].describe().round(1))


count    230820.0
mean       1714.3
std       10203.8
min     -118380.2
25%          17.0
50%          29.3
75%          50.1
max      180550.0
Name: kwh, dtype: float64


Find and drop duplicates (1/2)

How many traffic readings were sent twice, and how many rows are left after we drop them?



In [13]:
tr = pd.read_csv("data/lite/traffic_counts.csv",
                 parse_dates=["timestamp"])
key = ["sensor_id", "timestamp"]

dup = tr.duplicated(subset=key, keep=False)
print("rows sharing a key:", dup.sum())
display(tr[dup].sort_values(key).head(4))

before = len(tr)
tr = tr.drop_duplicates(subset=key, keep="first")
print(f"before {before:,} · after {len(tr):,}"
      f" · dropped {before - len(tr):,}")


rows sharing a key: 4792


,sensor_id,timestamp,vehicle_count,avg_speed_kmh,occupancy_pct
8804,TRF-001,2025-03-03 05:30:00,44.0,80.0,2.9
8805,TRF-001,2025-03-03 05:30:00,44.0,80.0,2.9
18988,TRF-001,2025-03-05 20:00:00,397.0,75.6,32.3
18989,TRF-001,2025-03-05 20:00:00,397.0,75.6,32.3


before 481,560 · after 479,164 · dropped 2,396


Resample on Cairo time

How do we turn 15-minute readings into hourly speeds, on a clock that knows daylight saving?


In [14]:
tr["ts"] = tr["timestamp"].dt.tz_localize(
    "Africa/Cairo", nonexistent="NaT", ambiguous="NaT")
print("readings in the hour that did not exist:",
      tr["ts"].isna().sum())
tr = tr.dropna(subset=["ts"])

hourly = (tr.set_index("ts").groupby("sensor_id")
            ["avg_speed_kmh"].resample("h").mean()
            .reset_index())
hourly["date"] = (hourly["ts"].dt.tz_localize(None)
                  .dt.normalize())          # plain local date
print(len(hourly), "sensor-hours")


readings in the hour that did not exist: 164
120007 sensor-hours


Group: weekday peak speed (1/2)

What is the average weekday peak speed of each traffic counter?

In [15]:
cal = pd.read_csv("reference/calendar.csv", parse_dates=["date"])
flags = cal[["date", "is_weekend", "is_holiday"]]
hourly = hourly.merge(flags, on="date", how="left",
                      validate="many_to_one")

# Peak (docs/kpi_reference.md): 07–09 and 15–18 on working days
work = (hourly["is_weekend"] == 0) & (hourly["is_holiday"] == 0)
in_peak = hourly["ts"].dt.hour.isin([7, 8, 15, 16, 17])
peak = hourly[work & in_peak]

peak_speed = (peak.groupby("sensor_id")["avg_speed_kmh"]
                  .mean().round(1))
print(peak_speed.head(3))
print("slowest:", peak_speed.idxmin(), peak_speed.min())


sensor_id
TRF-001    71.6
TRF-002    35.3
TRF-003    38.0
Name: avg_speed_kmh, dtype: float64
slowest: TRF-004 27.9


Questions · Tasks 1–13

Step 1 · Clean traffic_counts

Task 1: What period does each dataset cover, how many values are missing, and how many break the valid range in the data dictionary?


In [16]:
checks = [("traffic", tr_raw, "avg_speed_kmh", 120),
          ("electricity", el_raw, "kwh", 600)]
for name, df, col, hi in checks:   # valid range: 0 to hi
    print(name, df["timestamp"].min(), "→",
          df["timestamp"].max())
    print("   missing:", df[col].isna().sum(),
          "· below 0:", (df[col] < 0).sum(),
          f"· above {hi}:", (df[col] > hi).sum())


traffic 2025-03-01 00:00:00 → 2025-06-30 23:45:00
   missing: 3555 · below 0: 0 · above 120: 0
electricity 2025-03-01 00:00:00 → 2025-06-30 23:00:00
   missing: 3340 · below 0: 104 · above 600: 8785


Task 2: How many traffic readings repeat the same sensor and timestamp? Drop them and log the number

In [17]:
tr = tr_raw.copy()
key = ["sensor_id", "timestamp"]

n = tr.duplicated(subset=key).sum()
tr = tr.drop_duplicates(subset=key, keep="first")
report.append(["duplicate rows", "traffic_counts", n,
               "dropped, kept the first copy"])
print("duplicates dropped:", n, "· rows left:", len(tr))


duplicates dropped: 2396 · rows left: 479164


Task 3: Which readings are stamped in the hour that did not exist in Cairo on 25 April 2025?


In [18]:
local = tr["timestamp"].dt.tz_localize(
    "Africa/Cairo", nonexistent="NaT", ambiguous="NaT")
n = local.isna().sum()
ghost = tr.loc[local.isna(), "timestamp"]
times = ghost.dt.strftime("%d %b %H:%M").drop_duplicates()
print(times.tolist())

tr = tr[local.notna()]
report.append(["readings in the hour that did not exist",
               "traffic_counts", n, "dropped and logged"])
print("rows dropped:", n)


['25 Apr 00:00', '25 Apr 00:15', '25 Apr 00:30', '25 Apr 00:45']
rows dropped: 164


Task 4
Which sensor_id values are not in reference/sensors.csv, and how many rows do they have?


In [19]:
known = set(sensors["sensor_id"])
orphan = sorted(set(tr["sensor_id"]) - known)
tr["registered"] = tr["sensor_id"].isin(known)
n = (~tr["registered"]).sum()

report.append([f"sensor_id not in the registry {orphan}",
               "traffic_counts", n,
               "kept and flagged; left out of zone KPIs"])
print(orphan, "·", n, "rows")


['TRF-041'] · 11708 rows


Task 5a
Which counters stopped sending for 3 hours or more, and when did they come back?


In [20]:
t = tr.sort_values(["sensor_id", "timestamp"])
gap = t.groupby("sensor_id")["timestamp"].diff()
long = gap >= pd.Timedelta(hours=3)

blocks = t.loc[long, ["sensor_id", "timestamp"]].rename(
    columns={"timestamp": "back_at"})
blocks["hours_missing"] = gap[long].dt.total_seconds() / 3600
display(blocks)


,sensor_id,back_at,hours_missing
34651,TRF-005,2025-03-09 20:00:00,12.25
148246,TRF-005,2025-04-07 15:00:00,48.25
399712,TRF-005,2025-06-10 07:00:00,5.25
185038,TRF-012,2025-04-16 23:00:00,20.25
251795,TRF-012,2025-05-03 20:00:00,3.25
458349,TRF-012,2025-06-25 03:00:00,8.25
33389,TRF-029,2025-03-09 12:00:00,30.25
289809,TRF-029,2025-05-13 11:00:00,20.25
342571,TRF-029,2025-05-26 19:00:00,3.25
17377,TRF-036,2025-03-05 10:00:00,30.25


Task 5b
How many readings are absent, and what do we decide?


In [21]:
absent = int((gap[long] / pd.Timedelta(minutes=15) - 1).sum())
report.append([f"outage blocks of 3 h or more ({len(blocks)})",
               "traffic_counts", absent,
               "not filled; coverage reported"])
print(len(blocks), "blocks ·", absent, "readings absent")


12 blocks · 1028 readings absent


Task 6: How many speeds are missing here and there, and which can we fill safely?


In [24]:
from pathlib import Path
MY = Path("/content/drive/MyDrive")
OUT = (MY / "ACUD_Oasis" if MY.exists() else Path("outputs")) / "D02"
OUT.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUT)

tr.to_csv(OUT / "cleaned_traffic.csv", index=False)
print("missing:", n, "· filled:", tr["speed_filled"].sum())

Saving to: outputs/D02


KeyError: 'speed_filled'

In [26]:
from pathlib import Path
MY = Path("/content/drive/MyDrive")
OUT = (MY / "ACUD_Oasis" if MY.exists() else Path("outputs")) / "D02"
OUT.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUT)

tr.to_csv(OUT / "cleaned_traffic.csv", index=False)

remaining = tr["speed"].isna().sum()
print("missing:", n, "· filled:", n - remaining, "· still missing:", remaining)

Saving to: outputs/D02


KeyError: 'speed'

Step 2 · Clean electricity_meters

Task 7
Which buildings report in Wh instead of kWh? Convert only those buildings.


In [ ]:
el = el_raw.copy()
med = el.groupby("building_id")["kwh"].median()
wh = med[med > 600].index.tolist()   # typical hour above range

el.loc[el["building_id"].isin(wh), "kwh"] /= 1000
n = el["building_id"].isin(wh).sum()
report.append([f"Wh instead of kWh {wh}", "electricity_meters",
               n, "divided by 1,000 (these buildings only)"])
print(wh, "·", n, "rows converted")


Task 8
After the unit fix, which readings are still impossible? Set them to missing and flag them.


In [ ]:
bad = (el["kwh"] < 0) | (el["kwh"] > 600)
el["kwh_flag"] = np.where(bad, "impossible", "")
el.loc[bad, "kwh"] = np.nan
report.append(["impossible kWh (below 0 or above 600)",
               "electricity_meters", bad.sum(),
               "set to missing, flagged"])

miss = el_raw["kwh"].isna().sum()
report.append(["missing kWh", "electricity_meters", miss,
               "left missing; a day needs 20 hours or more"])
el.to_csv(OUT / "cleaned_electricity.csv", index=False)
print("impossible:", bad.sum(), "· missing:", miss)


Step 3 · Two KPIs and the report

Task 9 · demo
How much does cleaning change the daily kWh of one Wh building?



In [ ]:
def daily_kwh(df):
    return (df.set_index("timestamp")
              .groupby("building_id")["kwh"]
              .resample("D").sum(min_count=20)
              .rename("daily_kwh").reset_index())

raw_daily, clean_daily = daily_kwh(el_raw), daily_kwh(el)
b = wh[0]
side = raw_daily[raw_daily["building_id"] == b].merge(
    clean_daily, on=["building_id", "timestamp"],
    suffixes=("_raw", "_clean"))
side.head(3)


Helper · zone of each counter

Which zone is each traffic counter in?



In [ ]:
# Provided: you will write this join yourself tomorrow (Day 3)
sensor_zone = sensors.set_index("sensor_id")["zone_id"]

def add_zone(df):
    return df.assign(zone_id=df["sensor_id"].map(sensor_zone)

Task 10a

How do we turn every counter's 15-minute speeds into hourly speeds on Cairo time?


In [ ]:
def hourly(df):     # Session 2: Cairo clock, hourly mean, date
    ts = df["timestamp"].dt.tz_localize(
        "Africa/Cairo", nonexistent="NaT", ambiguous="NaT")
    d = df.assign(ts=ts).dropna(subset=["ts"])
    h = (d.set_index("ts").groupby("sensor_id")["avg_speed_kmh"]
          .resample("h").mean().reset_index())
    h["date"] = h["ts"].dt.tz_localize(None).dt.normalize()
    return h

print(len(hourly(tr)), "sensor-hours")


Task 10b

What is the average weekday peak speed in each zone, on normal days and in Ramadan?


In [ ]:
def peak_speed_by_zone(df):
    h = add_zone(hourly(df).merge(cal, on="date", how="left"))
    peak = ((h["is_weekend"] == 0) & (h["is_holiday"] == 0)
            & h["ts"].dt.hour.isin([7, 8, 15, 16, 17]))
    return (h[peak].groupby(["zone_id", "is_ramadan"])
              ["avg_speed_kmh"].mean().unstack().round(1)
              .rename(columns={0: "normal", 1: "ramadan"}))

kpi1 = peak_speed_by_zone(tr_raw).join(
    peak_speed_by_zone(tr), lsuffix="_raw", rsuffix="_clean")
kpi1


Task 11
How many building-days change their daily kWh after cleaning?


In [ ]:
kpi2 = raw_daily.merge(clean_daily,
                       on=["building_id", "timestamp"],
                       suffixes=("_raw", "_clean"))
diff = (kpi2["daily_kwh_raw"] - kpi2["daily_kwh_clean"]).abs()
changed = kpi2[diff > 0.5]
print(f"{len(changed):,} of {len(kpi2):,}",
      "building-days changed")
print(changed.groupby("building_id").size()
             .sort_values(ascending=False).head(4))


Task 12a
How did the traffic decisions move the numbers?

In [ ]:
veh = lambda d: (d.groupby(d["timestamp"].dt.normalize())
                  ["vehicle_count"].sum().mean())
coverage = 1 - absent / (len(tr) + absent)
print(f"vehicles per day: {veh(tr_raw):,.0f} → {veh(tr):,.0f}")
print(f"KPI 1 coverage: {coverage:.1%}")


Task 12b 
Did filling speeds or leaving kWh blank change the KPIs?


In [ ]:
unfilled = tr.assign(avg_speed_kmh=tr["avg_speed_kmh"]
                     .mask(tr["speed_filled"]))
moved = (peak_speed_by_zone(unfilled)
         - peak_speed_by_zone(tr)).abs().max().max()
raw_day, clean_day = side[["daily_kwh_raw",
                           "daily_kwh_clean"]].median()
blank = clean_daily["daily_kwh"].isna().sum()
print(f"filled speeds move KPI 1 by {moved:.1f} km/h")
print(f"daily kWh left blank (under 20 hours): {blank}")


Task 13
What did we find, what did we decide, and how did it change the KPIs?


In [ ]:
rep = pd.DataFrame(report, columns=["issue", "dataset",
                                   "rows affected", "decision"])
rep["impact on KPI"] = [
    f"vehicles per day {veh(tr_raw):,.0f} → {veh(tr):,.0f}",
    "none: 00:00–00:59 is outside the peak hours",
    "none: no zone, so it is not in the zone KPIs",
    f"KPI 1 uses {coverage:.1%} of the expected readings",
    f"KPI 1 moves by {moved:.1f} km/h at most",
    f"{b}: median day {raw_day:,.0f} → {clean_day:,.1f} kWh",
    f"{len(changed):,} building-days change (with the unit fix)",
    f"{blank} building-days left blank (under 20 hours)"]
rep.to_excel(OUT / "D02_data_quality_report.xlsx", index=False)
kpi1.to_csv(OUT / "kpi_peak_speed_by_zone.csv")
clean_daily.to_csv(OUT / "kpi_daily_kwh.csv", index=False)
display(rep)


Prove it · Same key, same row?

Are the duplicates exact copies, or the same key with different values?

In [ ]:
key = ["sensor_id", "timestamp"]
print("same key:", tr_raw.duplicated(subset=key).sum())
print("same row:", tr_raw.duplicated().sum())


Prove it · The 23-hour day

Is 25 April an outage for the electricity meters, or a short day?



In [ ]:
days = pd.to_datetime(["2025-04-24", "2025-04-25", "2025-04-26"])
three = el_raw[el_raw["timestamp"].dt.normalize().isin(days)]
rows = three.groupby([three["timestamp"].dt.date,
                      "building_id"]).size()
print(rows.groupby(level=0).agg(["min", "max"]))


Prove it · The unit fix

After dividing by 1,000, do the three buildings look like the rest of the district?


In [ ]:
med = el.groupby("building_id")["kwh"].median()
print(med[wh].round(1))
print("district median:", round(med.median(), 1))


Challenge 1 · A reusable cleaning function

Can you turn Tasks 2–4 into one function that cleans any traffic file the same way?

Then: Does the function give exactly the rows you got step by step?

Hint: Copy the steps of Tasks 2–4 into one function that takes the raw table and the registry and returns the clean table.


Challenge 1a · Solution

Can you turn Tasks 2–4 into one function that cleans any traffic file the same way?

Challenge 1b · Solution

Does the function give exactly the rows you got step by step?


Challenge 2 · A third KPI: daily kWh by zone

What is the average daily kWh of a metered building in each zone, on raw and on clean data?

Hint: Add the zone to raw_daily and clean_daily from reference/buildings.csv, then average per zone.
